# Analyze the spatial corr and the overlap characteristics of the "supportive" and "antergic" networks of rumination
## Xiao Chen 260504
## chenxiaophd@gmail.com

In [ ]:
import numpy as np
import pandas as pd
import neuromaps
from neuromaps import stats
from neuromaps import datasets, images, nulls, resampling
from neuromaps.datasets import available_annotations
from neuromaps.datasets import fetch_annotation
from neuromaps.resampling import resample_images
import os
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
from nilearn import plotting, datasets

In [ ]:
work_dir = "xxxxxxxxxxxxx"
out_dir = os.path.join(work_dir, "overlap_analysis")
atlas_dir = "xxxxxxxxxxx"

## to get the "supportive" and "antergic" networks of rumination

In [ ]:
map_label = "z_rumination_MDD_GSR"
# get the gii files
left_gii = f"~/rumination_lesion_network_mapping/maps/{map_label}_lh.gii")
right_gii = f"~/rumination_lesion_network_mapping/maps/{map_label}_rh.gii"


# ====================================
# 2. read the right hemisphere GIFTI
# ====================================
lh_img = nib.load(left_gii)
rh_img = nib.load(right_gii)

lh_data = lh_img.darrays[0].data.copy()
rh_data = rh_img.darrays[0].data.copy()

# ====================================
# 3. get the positive and negative parts
# ====================================
lh_pos = lh_data.copy()
lh_pos[lh_pos <= 0] = 0

lh_neg = lh_data.copy()
lh_neg[lh_neg >= 0] = 0

rh_pos = rh_data.copy()
rh_pos[rh_pos <= 0] = 0

rh_neg = rh_data.copy()
rh_neg[rh_neg >= 0] = 0

# ====================================
# 4. save out as new .gii files
# ====================================
lh_pos_img = nib.gifti.GiftiImage(
    darrays=[nib.gifti.GiftiDataArray(lh_pos)]
)

lh_neg_img = nib.gifti.GiftiImage(
    darrays=[nib.gifti.GiftiDataArray(lh_neg)]
)

rh_pos_img = nib.gifti.GiftiImage(
    darrays=[nib.gifti.GiftiDataArray(rh_pos)]
)

rh_neg_img = nib.gifti.GiftiImage(
    darrays=[nib.gifti.GiftiDataArray(rh_neg)]
)

# ====================================
# 5. output files
# ====================================
lh_pos_file = os.path.join(out_dir, f"{map_label}_lh_pos.gii")
lh_neg_file = os.path.join(out_dir, f"{map_label}_lh_neg.gii")
rh_pos_file = os.path.join(out_dir, f"{map_label}_rh_pos.gii")
rh_neg_file = os.path.join(out_dir, f"{map_label}_rh_neg.gii")

nib.save(lh_pos_img, lh_pos_file)
nib.save(lh_neg_img, lh_neg_file)
nib.save(rh_pos_img, rh_pos_file)
nib.save(rh_neg_img, rh_neg_file)

## take a peek

In [ ]:
gii = nib.load(os.path.join(out_dir, f"{map_label}_lh_neg.gii"))
data = gii.darrays[0].data

# fsaverage surface
fsaverage = datasets.fetch_surf_fsaverage()

# look at them in the browser
view = plotting.view_surf(
    surf_mesh=fsaverage.infl_left,
    surf_map=data,
    bg_map=fsaverage.sulc_left,
    cmap="seismic"
)

view.open_in_browser()

## function: calculate the dice coefficient

In [ ]:
def dice_gii_bilateral(lh_gii_1, rh_gii_1, lh_gii_2, rh_gii_2):
    """
    calculte the Dice coefficients from two pairs of .gii files

    Parameters
    ----------
    lh_gii_1, rh_gii_1 : str 
        pair 1

    lh_gii_2, rh_gii_2 : str
        pair 2

    Returns
    -------
    dice : float
    """

    # read in .gii files
    lh_1 = nib.load(lh_gii_1).darrays[0].data
    rh_1 = nib.load(rh_gii_1).darrays[0].data

    lh_2 = nib.load(lh_gii_2).darrays[0].data
    rh_2 = nib.load(rh_gii_2).darrays[0].data

    # conbine two hemisphere
    data_1 = np.concatenate([lh_1, rh_1])
    data_2 = np.concatenate([lh_2, rh_2])

    # non-zero mask
    mask_1 = data_1 != 0
    mask_2 = data_2 != 0

    # Dice
    intersection = np.sum(mask_1 & mask_2)
    size_1 = np.sum(mask_1)
    size_2 = np.sum(mask_2)

    if size_1 + size_2 == 0:
        dice = np.nan
    else:
        dice = 2 * intersection / (size_1 + size_2)

    return dice

In [ ]:
networks = ['VN','SMN','DAN','VAN','LN','FPN','DMN']
lh_gii_1 = os.path.join(out_dir, f"{map_label}_lh_neg.gii")
rh_gii_1 = os.path.join(out_dir, f"{map_label}_rh_neg.gii")
dice_values = []
for net in networks:
    lh_gii_2 = os.path.join(atlas_dir, f"{net}_lh.gii")
    rh_gii_2 = os.path.join(atlas_dir, f"{net}_rh.gii")
    d = dice_gii_bilateral(
        lh_gii_1, rh_gii_1,
        lh_gii_2, rh_gii_2
    )
    dice_values.append(d)

dice_values_neg = np.array(dice_values)

In [ ]:
plt.figure(figsize=(6, 4), dpi=300)

bars = plt.bar(networks, dice_values_neg)

plt.ylabel("Dice coefficient")
plt.title("Overlap with rumination map")

for i, v in enumerate(dice_values_neg):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center', fontsize=10)

plt.ylim(0, max(dice_values_neg) * 1.2)

plt.tight_layout()
plt.show()

In [ ]:
networks = ['VN','SMN','DAN','VAN','LN','FPN','DMN']
lh_gii_1 = os.path.join(out_dir, f"{map_label}_lh_pos.gii")
rh_gii_1 = os.path.join(out_dir, f"{map_label}_rh_pos.gii")
dice_values = []
for net in networks:
    lh_gii_2 = os.path.join(atlas_dir, f"{net}_lh.gii")
    rh_gii_2 = os.path.join(atlas_dir, f"{net}_rh.gii")
    d = dice_gii_bilateral(
        lh_gii_1, rh_gii_1,
        lh_gii_2, rh_gii_2
    )
    dice_values.append(d)

dice_values_pos = np.array(dice_values)

In [ ]:
plt.figure(figsize=(6, 4), dpi=300)

bars = plt.bar(networks, dice_values_pos)

plt.ylabel("Dice coefficient")
plt.title("Overlap with rumination map")

# 可选：数值标注
for i, v in enumerate(dice_values_pos):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center', fontsize=10)

plt.ylim(0, max(dice_values_pos) * 1.2)

plt.tight_layout()
plt.show()

In [ ]:
# save out

vec1 = dice_values_neg
vec2 = dice_values_pos

df = pd.DataFrame({
    "Network": networks,
    "Negative": vec1,
    "Positive": vec2
})

df.to_csv(os.path.join(out_dir, "network_dice.csv"), index=False)